# Modelación en ingeniería
## Diseño de señales de entrada

*Profesor: David Ortiz-Puerta*

---

En identificación fenomenológica, no conocemos la dinámica interna del sistema. Solo disponemos de mediciones entrada-salida $\{u[n], y[n]\}_{n=1}^{N}$ y proponemos una estructura que capture la relación entre ellas.

En este notebook aplicamos dos métodos de estimación sobre datos sintéticos del sistema Lotka-Volterra:

- **LS (Mínimos Cuadrados):** estimación batch, usa todos los datos a la vez.
- **RLS (Mínimos Cuadrados Recursivo):** estimación en línea, actualiza $\hat{\theta}$ dato a dato.

Ambos métodos operan sobre la estructura **ARX**, construida exclusivamente con observables: salidas y entradas pasadas.

> **💡 Nota para usuarios de Google Colab**
> 
> Antes de ejecutar el notebook, descarga las funciones auxiliares ejecutando la siguiente celda

In [ ]:
# !wget -q https://raw.githubusercontent.com/dortiz5/modelacion-en-ingenieria/main/Notebooks/helpers.py
# !wget -q https://raw.githubusercontent.com/dortiz5/modelacion-en-ingenieria/main/data/lotka_volterra/lotka_volterra_20260514_003.csv

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from helpers import (LS, RLS, configure_plot_style, analisis_residual)

configure_plot_style()

### Construcción de la matriz de regresores $\Phi$

El modelo ARX-SISO propone que la salida actual depende de salidas y entradas pasadas:

$$y[n] = -a_1 y[n-1] - \cdots - a_{n_a} y[n-n_a] + b_1 u[n-1] + \cdots + b_{n_b} u[n-n_b] + e[n]$$

Definimos el vector de regresión $\varphi[n]$ y el vector de parámetros $\theta$:

$$\varphi[n] = \begin{bmatrix} -y[n-1] & \cdots & -y[n-n_a] & u[n-1] & \cdots & u[n-n_b] \end{bmatrix}^T \in \mathbb{R}^{n_a+n_b}$$

$$\theta = \begin{bmatrix} a_1 & \cdots & a_{n_a} & b_1 & \cdots & b_{n_b} \end{bmatrix}^T \in \mathbb{R}^{n_a+n_b}$$

Apilando $N - n_0$ ecuaciones (donde $n_0 = \max(n_a, n_b)$ es el primer índice válido):

$$\underbrace{\begin{bmatrix} y[n_0] \\ y[n_0+1] \\ \vdots \\ y[N] \end{bmatrix}}_{y \in \mathbb{R}^{N-n_0}} = \underbrace{\begin{bmatrix} \varphi^T[n_0] \\ \varphi^T[n_0+1] \\ \vdots \\ \varphi^T[N] \end{bmatrix}}_{\Phi \in \mathbb{R}^{(N-n_0)\times(n_a+n_b)}} \theta + \underbrace{\begin{bmatrix} e[n_0] \\ e[n_0+1] \\ \vdots \\ e[N] \end{bmatrix}}_{e \in \mathbb{R}^{N-n_0}}$$

Los parámetros $n_a$ y $n_b$ son parámetros de diseño — el ingeniero los elige. Un modelo más rico ($n_a$, $n_b$ grandes) no siempre es mejor.


In [ ]:
def construir_phi(y, u, na = 1, nb = 1):
    
    N  = len(y)
    n0 = max(na, nb)
    M  = N - n0

    Phi   = np.zeros((M, na + nb))
    y_vec = np.zeros(M)

    for i, n in enumerate(range(n0, N)):
        # Retardos de salida: -y[n-1], ..., -y[n-na]
        for j in range(na):
            Phi[i, j] = -y[n - 1 - j]
        # Retardos de entrada: u[n-1], ..., u[n-nb]
        for j in range(nb):
            Phi[i, na + j] = u[n - 1 - j]
        y_vec[i] = y[n]

    return Phi, y_vec, n0

### Estimación LS y RLS — Comparación

Comparamos ambos estimadores sobre los mismos datos. LS resuelve el problema batch con todos los datos a la vez. RLS actualiza $\hat{\theta}[n]$ recursivamente cada vez que llega un nuevo dato.

$$\hat{\theta}_{LS} = \left(\Phi^T\Phi\right)^{\dagger}\Phi^T y \qquad \hat{\theta}_{RLS}[n] = \hat{\theta}[n-1] + K[n]\left(y[n] - \varphi^T[n]\hat{\theta}[n-1]\right)$$

Los parámetros de RLS son: factor de olvido $\lambda \in (0,1]$ e incertidumbre inicial $P[0] = \alpha I$.


In [ ]:
# --- Parámetros ---
na      = 2
nb      = 2
lam     = 0.98
alpha   = 1e6
especie = 'x1'     # 'x1' o 'x2'

# --- Carga de datos ---
name = f'lotka_volterra_20260514_003.csv'
df = pd.read_csv(name)
t  = df['t'].values
u  = df['u'].values
y  = df[especie].values

# --- Construcción de Φ ---
Phi, y_vec, n0 = construir_phi(y, u, na, nb)
t_valido = t[n0:n0+len(y_vec)]   # mismo largo que y_vec garantizado

print(f"Parámetros de diseño: na = {na}, nb = {nb}")
print(f"n0 = {n0} (primer índice válido)")
print(f"Φ ∈ R^({Phi.shape[0]} x {Phi.shape[1]})")
print(f"θ ∈ R^{na+nb}")
print(f"\nPrimeras 5 filas de Φ:")
print(f"[-y[n-1], -y[n-2], u[n-1], u[n-2]]")
print("-" * 45)
for i in range(5):
    print(f"fila {i+n0:>3}: {Phi[i]}")
    
    
# --- Estimación ---
theta_LS,  y_hat_LS,  res_LS            = LS(Phi, y_vec)
theta_RLS, y_hat_RLS, res_RLS, hist_RLS = RLS(Phi, y_vec, lam=lam, alpha=alpha)

print(f"\n{'Parámetro':>12} {'LS':>10} {'RLS':>10}")
print("-" * 34)
for i in range(na):
    print(f"{'a'+str(i+1):>12} {theta_LS[i]:>10.4f} {theta_RLS[i]:>10.4f}")
for i in range(nb):
    print(f"{'b'+str(i+1):>12} {theta_LS[na+i]:>10.4f} {theta_RLS[na+i]:>10.4f}")

In [ ]:
# --- Visualización ---
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

axes[0].plot(t_valido, y_vec,     color='gray',      alpha=0.4, label='Medición ruidosa')
axes[0].plot(t_valido, y_hat_LS,  color='steelblue', lw=1.5, ls='--', label='LS')
axes[0].plot(t_valido, y_hat_RLS, color='darkorange', lw=1.5, ls='-.', label='RLS')
axes[0].set_ylabel(f'${especie}$')
axes[0].set_title('Predicción')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_valido, res_LS,  color='steelblue',  lw=1.0, label='Residuos LS')
axes[1].plot(t_valido, res_RLS, color='darkorange',  lw=1.0, label='Residuos RLS')
axes[1].axhline(0, color='black', lw=0.8, ls='--')
axes[1].set_ylabel('Residuos')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

for i in range(na):
    axes[2].plot(t_valido, hist_RLS[:, i], lw=1.5, label=rf'$\hat{{a}}_{i+1}$ RLS')
    axes[2].axhline(theta_LS[i],ls='--', alpha=0.6, label=rf'$\hat{{a}}_{i+1}$ LS')
for i in range(nb):
    axes[2].plot(t_valido, hist_RLS[:, na+i], lw=1.5, label=rf'$\hat{{b}}_{i+1}$ RLS')
    axes[2].axhline(theta_LS[na+i], ls='--', alpha=0.6, label=rf'$\hat{{b}}_{i+1}$ LS')
    
axes[2].set_title('Convergencia de parámetros')
axes[2].set_xlabel('t [s]')
axes[2].set_ylabel(r'$\hat{\theta}$')
axes[2].legend(fontsize=8, ncol=2)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Validación: análisis residual

Un modelo bien identificado debe capturar toda la dinámica relevante del sistema. Si esto se cumple, el residuo $\varepsilon[n] = y[n] - \hat{y}[n]$ debe comportarse como ruido blanco no correlacionado con la entrada.

Esto se verifica mediante dos tests:
- **Autocorrelación** $R_\varepsilon(\tau)$: debe ser aproximadamente cero para $\tau \neq 0$.
- **Correlación cruzada** $R_{\varepsilon u}(\tau)$: debe ser aproximadamente cero para todo $\tau$.

Las bandas de confianza $\pm 2/\sqrt{N}$ indican el umbral estadístico. Valores fuera de banda señalan dinámica no capturada por el modelo.

**Actividad:** modifica los parámetros $n_a$ y $n_b$ en el bloque anterior hasta que los residuos de ambos métodos queden dentro de las bandas. ¿Qué ocurre si aumentas demasiado el orden? ¿Y si es muy pequeño?

In [ ]:
analisis_residual(res_LS,  u[n0:n0+len(res_LS)],  t_valido, label='LS')
analisis_residual(res_RLS, u[n0:n0+len(res_RLS)], t_valido, label='RLS')